# Ścieżka D · Framing i stabilność wyboru

**Projektowanie i badanie informatycznych systemów wspomagania decyzji** · notatnik projektu z aplikacją · szkielet do uzupełnienia

Notatnik prowadzi cały projekt ścieżki D w jednym miejscu, zgodnie z przewodnikiem *D — Framing — Przewodnik*:

| Etap | Kto robi | Pliki |
|---|---|---|
| 1–3 · opisy z jednego rekordu, eksport, FI | zespół (luki `# TODO`) | `qualtrics_raw.csv` → `responses_clean.csv` → `profil.csv` |
| 4 · nowe przypadki (12 wersji) | zespół (luki `# TODO`) | `cases.csv` |
| 5 · zamrożenie, kontrakt, aplikacja | gotowe | `session_config.json`, `app.py` |
| 6 · wyniki walidacji | zespół | wynik do raportu |

**Aplikacja jest gotowa.** Nie piszecie interfejsu. Aplikacja czyta pliki tworzone przez ten notatnik i działa tylko wtedy, gdy spełniają one kontrakt.

**Dwa tryby pracy** (komórka konfiguracji poniżej):
- `TRYB = 'przyklad'`: wymyślony uczestnik P01 z przewodnika. Testy sprawdzają liczby kontrolne z przewodnika. Zacznij tutaj.
- `TRYB = 'wlasne'`: eksport Waszego uczestnika. Testy funkcji działają dalej, a testy liczb P01 są pomijane: notatnik wypisuje wtedy Wasz wynik.

W tej ścieżce **nie liczymy accuracy**. Aplikacja nie przewiduje wyboru. Pokazuje równoważne opisy i zapisuje przebieg decyzji. Głównymi wynikami są udział i kierunek rewizji, rozumienie i transparentność. Rozbieżność par po etapie zbalansowanym jest tylko kontrolą.

> **Jak pracować.** Uruchamiaj komórki po kolei. W miejscach `# TODO` uzupełnij kod i usuń `raise NotImplementedError`. Test pod każdym krokiem pokazuje ✓ albo zatrzymuje się na ✗ z opisem. Nie zmieniaj liczb w testach.

## 0 · Konfiguracja i folder projektu na Dysku Google
W Colabie pliki znikają po rozłączeniu, dlatego folder roboczy leży na Dysku Google. Przy pierwszym uruchomieniu Colab poprosi o zgodę na dostęp do Dysku. Folder można udostępnić zespołowi i prowadzącym. Dla przykładu i dla każdego uczestnika powstaje osobny folder.

W plikach są tylko kod uczestnika i jego odpowiedzi. Informacja przed zgodą powinna mówić, gdzie przechowujecie dane i kto ma do nich dostęp.

In [ ]:
TRYB = 'przyklad'          # 'przyklad' albo 'wlasne'
PARTICIPANT = 'P01'        # kod uczestnika (taki sam w Qualtricsie i w sesji)
SESSION_ID = 'S01'         # identyfikator sesji walidacyjnej
INSTRUCTOR_PIN = '2468'    # kod widoku prowadzącego w aplikacji; nie pokazuj go uczestnikowi
UZYJ_DYSKU = True          # w Colabie: folder projektu na Dysku Google

import os, sys, hashlib, inspect, json, math
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
if TRYB not in ('przyklad', 'wlasne'):
    raise ValueError("TRYB musi mieć wartość 'przyklad' albo 'wlasne'")
if TRYB == 'przyklad':
    PARTICIPANT = 'P01'
if IN_COLAB and UZYJ_DYSKU:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/projekt_D'
else:
    ROOT = os.path.abspath('projekt_D')
WORKDIR = os.path.join(ROOT, 'przyklad_P01' if TRYB == 'przyklad' else PARTICIPANT)
os.makedirs(WORKDIR, exist_ok=True)
os.chdir(WORKDIR)
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 40)

def check(label, cond):
    """Test ogólny: działa w obu trybach."""
    if not bool(cond):
        raise AssertionError('✗ ' + label)
    print('✓', label)

def check_example(label, cond, value=None):
    """Test liczb przykładu P01: w trybie 'wlasne' tylko wypisuje Wasz wynik."""
    if TRYB == 'przyklad':
        check(label, cond)
    else:
        print('ℹ pominięto test przykładu:', label, '' if value is None else f'| Wasz wynik: {value}')

def close(a, b, tol=1e-6):
    return a is not None and b is not None and abs(a - b) <= tol

def now_iso():
    return datetime.now(timezone.utc).isoformat(timespec='seconds')

def file_sha256(path):
    return hashlib.sha256(open(path, 'rb').read()).hexdigest()

def write_qualtrics_csv(path, rows, descriptions):
    """Plik w układzie eksportu Qualtricsa: 3 wiersze nagłówkowe + odpowiedzi."""
    cols = list(rows[0].keys())
    df = pd.DataFrame([[descriptions.get(c, c) for c in cols], [json.dumps({'ImportId': f'QID_{c}'}) for c in cols]]
                      + [[r.get(c, '') for c in cols] for r in rows], columns=cols)
    df.to_csv(path, index=False)

print(f'Tryb: {TRYB} | uczestnik: {PARTICIPANT} | folder: {WORKDIR}')

## 1 · Rekordy problemów i równoważność ram (rozdz. 3–4)
Każdy problem to jeden rekord: punkty początkowe N i prawdopodobieństwo p zachowania całości. Opcja pewna zachowuje K = N · p. **Oba opisy generujemy z tego samego rekordu.** Równoważność sprawdzamy na stanach końcowych i ich prawdopodobieństwach. Uczestnik widzi opcje A i B. Kody SAFE/RISKY są wyłącznie techniczne.

In [ ]:
PROTOCOL = 'D_v1'
MODEL_VERSION = 'D_records_v1'
SAFE, RISKY = 1, 2                     # Recode Values w obu ramach
CODE = {1: 'SAFE', 2: 'RISKY'}
PROFILE_PAIRS = [('P01', 100, .20), ('P02', 100, .40), ('P03', 200, .25), ('P04', 200, .60), ('P05', 400, .75),
                 ('P06', 400, .10), ('P07', 500, .30), ('P08', 500, .70), ('P09', 1000, .45), ('P10', 1000, .80)]
VALIDATION_PAIRS = [('N01', 120, .25), ('N02', 150, .40), ('N03', 300, .60), ('N04', 600, .30), ('N05', 800, .75), ('N06', 1200, .80)]

def pct(p):
    return round(p * 100)

def describe(N, p, frame):
    """Opis w ramie 'G' (zachowanie) albo 'L' (utrata); opcja A = SAFE, B = RISKY."""
    K = round(N * p)
    # TODO: dwa szablony tekstu z przewodnika (rama zachowania i rama utraty)
    raise NotImplementedError

def outcomes(N, p, frame):
    """Stany końcowe {opcja: sorted[(punkty_końcowe, prawdopodobieństwo)]} wyprowadzone z opisu ramy."""
    K = round(N * p)
    # TODO: w ramie utraty zamień „stracisz x” na stan końcowy N − x; zwróć posortowane listy par
    raise NotImplementedError

def explanation(N, p):
    """Gotowe."""
    K = round(N * p)
    return (f'Początkowo masz {N} punktów. Zachować {K} to stracić {N - K}; zachować {N} to stracić 0. '
            'W obu opisach loteria ma te same szanse i te same końcowe punkty.')

check('100 punktów, p = 0,8: A zachowuje 80 = traci 20', 'zachowasz 80' in describe(100, .8, 'G') and 'stracisz 20' in describe(100, .8, 'L'))
check('zły stan loterii: 20% szans na utratę 100', '20% szans na utratę 100' in describe(100, .8, 'L'))
check('obie ramy mają te same stany końcowe dla 16 rekordów', all(outcomes(N, p, 'G') == outcomes(N, p, 'L') for _, N, p in PROFILE_PAIRS + VALIDATION_PAIRS))
print(describe(100, .8, 'G')); print(describe(100, .8, 'L')); print(explanation(100, .8))

## 2 · Eksport profilowania (rozdz. 5)
Pola `D_P01_G` … `D_P10_L`, z kodami 1 = SAFE i 2 = RISKY. Blok 1 zawiera P01 G, P02 L, P03 G i dalej naprzemiennie. Blok 2 zawiera odpowiedniki w tej samej kolejności, oddzielone zadaniem neutralnym.

In [ ]:
P01_CHOICES = {'P01': ('SAFE', 'RISKY'), 'P02': ('SAFE', 'SAFE'), 'P03': ('RISKY', 'RISKY'), 'P04': ('SAFE', 'RISKY'),
               'P05': ('RISKY', 'SAFE'), 'P06': ('SAFE', 'SAFE'), 'P07': ('RISKY', 'RISKY'), 'P08': ('SAFE', 'RISKY'),
               'P09': ('SAFE', 'SAFE'), 'P10': ('RISKY', 'RISKY')}
FIELDS_P01 = {f'D_{pid}_{f}': {'SAFE': SAFE, 'RISKY': RISKY}[P01_CHOICES[pid][k]] for pid, *_ in PROFILE_PAIRS for k, f in enumerate('GL')}

def make_example_raw(path):
    base = dict(ResponseId='R_P01', participant_id='P01', project_path='D', protocol_version=PROTOCOL, phase='profile',
                Status=0, Finished=1, Progress=100, **{'Duration (in seconds)': 402}, zgoda=1)
    desc = {f'D_{pid}_{f}': describe(N, p, f) for pid, N, p in PROFILE_PAIRS for f in 'GL'}
    write_qualtrics_csv(path, [{**base, 'ResponseId': 'R_test01', 'Status': 2, **FIELDS_P01}, {**base, **FIELDS_P01}], desc)

In [ ]:
RAW_PATH = 'qualtrics_raw.csv'
if TRYB == 'przyklad':
    make_example_raw(RAW_PATH)
elif not os.path.exists(RAW_PATH):
    if not IN_COLAB:
        raise FileNotFoundError(f'Umieść eksport jako {os.path.join(WORKDIR, RAW_PATH)}')
    from google.colab import files
    up = files.upload()
    open(RAW_PATH, 'wb').write(next(iter(up.values())))
print('Plik źródłowy:', os.path.join(WORKDIR, RAW_PATH))

In [ ]:
def load_raw(path):
    """Eksport Qualtricsa: trzy wiersze nagłówkowe, dane od czwartego."""
    return pd.read_csv(path, skiprows=[1, 2])

def select_response(raw, participant_id, protocol_version, phase='profile'):
    """Dokładnie jedno wypełnienie: Status 0 (nie Preview 1, nie Test 2), Finished 1, zgoda 1, kod, wersja, faza."""
    # TODO: przefiltruj raw według sześciu warunków z opisu, sprawdź, że został dokładnie jeden wiersz, zwróć go
    raise NotImplementedError

raw = load_raw(RAW_PATH)
resp = select_response(raw, PARTICIPANT, PROTOCOL)
print('Wybrano wypełnienie', resp['ResponseId'])

## 3 · Długi plik i Framing Index (rozdz. 6)
$FI = \frac{\text{pary ze zmianą wyboru}}{\text{ważne pary}}$. Para jest ważna, gdy ma odpowiedź w obu ramach. Raportujemy też kierunki zmian. Bez ważnych par FI jest **nieustalone** (`None`), a nie równe 0. FI opisuje rozbieżności w tym zestawie problemów. Nie jest cechą osoby ani odsetkiem błędów.

In [ ]:
def to_long(row):
    """20 wierszy: pair_id, frame (GAIN/LOSS), endowment, success_p, safe_kept, choice (SAFE/RISKY/None)."""
    # TODO: po dwa wiersze na parę; zamień kody 1/2 na SAFE/RISKY, brak zostaw jako None
    raise NotImplementedError

def framing_index(long):
    """dict(n_valid, n_changed, fi, gain_safe_to_loss_risky, gain_risky_to_loss_safe)."""
    # TODO: tabela szeroka pair_id × frame, ważne pary (obie odpowiedzi), zmiany i ich kierunki
    raise NotImplementedError

_long = to_long(FIELDS_P01)
_fi = framing_index(_long)
check('dane P01: FI = 4/10, kierunki 3 i 1', (_fi['n_valid'], _fi['n_changed'], _fi['gain_safe_to_loss_risky'], _fi['gain_risky_to_loss_safe']) == (10, 4, 3, 1))
check('brak P05 LOSS → FI = 3/9', (lambda f: (f['n_valid'], f['n_changed']))(framing_index(to_long({**FIELDS_P01, 'D_P05_L': ''}))) == (9, 3))
check('brak ważnych par → FI nieustalone', framing_index(_long.assign(choice=None))['fi'] is None)
try:
    framing_index(pd.concat([_long, _long.iloc[[0]]])); check('duplikat wersji odrzucony', False)
except ValueError:
    check('duplikat wersji odrzucony', True)

long = to_long(resp)
long.to_csv('responses_clean.csv', index=False)
fi = framing_index(long)
check_example('FI P01 = 0,4', fi['fi'] == 0.4, fi)
pd.DataFrame([dict(participant_id=PARTICIPANT, protocol_version=PROTOCOL, **fi)]).to_csv('profil.csv', index=False)
fi

## 4 · Nowe przypadki: 12 wersji i kolejność etapów (rozdz. 9–10)
Najpierw zbieramy **wszystkie** 12 wyborów standardowych (`initial`). Dopiero potem etap zbalansowany pokazuje dla tych samych `case_id` oba opisy z wyjaśnieniem i zbiera `final`.

In [ ]:
STANDARD_ORDER = [('N01', 'G'), ('N02', 'L'), ('N03', 'G'), ('N04', 'L'), ('N05', 'G'), ('N06', 'L'),
                  ('N01', 'L'), ('N02', 'G'), ('N03', 'L'), ('N04', 'G'), ('N05', 'L'), ('N06', 'G')]
REC = {pid: (N, p) for pid, N, p in VALIDATION_PAIRS}

def build_cases():
    """12 wierszy: order, pair_id, case_id, origin_frame, endowment, success_p, safe_kept, safe_lost,
    text_standard, text_gain, text_loss, explanation."""
    # TODO: wiersz dla każdej pozycji STANDARD_ORDER; teksty z describe() i explanation()
    raise NotImplementedError

cases = build_cases()
check('12 case_id, każda para w obu ramach', len(cases) == 12 and cases.groupby('pair_id').origin_frame.nunique().eq(2).all())
check('N01: zachowane 30 / utracone 90; N06: 960 / 240', cases.query('case_id == "N01_G"')[['safe_kept', 'safe_lost']].values.tolist() == [[30, 90]]
      and cases.query('case_id == "N06_L"')[['safe_kept', 'safe_lost']].values.tolist() == [[960, 240]])
check('każdy case_id ma równoważne ramy', all(outcomes(r.endowment, r.success_p, 'G') == outcomes(r.endowment, r.success_p, 'L') for r in cases.itertuples()))
COMPREHENSION = [dict(pair_id='N02', question='Ile punktów zostanie, jeśli w opcji B los wypadnie niekorzystnie?', correct=0),
                 dict(pair_id='N05', question='Ile punktów zostanie na pewno w opcji A?', correct=600)]
PROFILE_OK, PROFILE_NOTE = True, 'FI jest opisem; aplikacja jest dostępna dla każdego uczestnika'
SESSION_EXTRA = dict(title='Wybory punktowe', comprehension=COMPREHENSION,
                     intro='W kolejnych hipotetycznych sytuacjach wybierz opcję, którą wolisz. Opcja A daje wynik pewny, a opcja B '
                           'wynik niepewny o podanych szansach. Zwracaj uwagę na punkty początkowe i możliwe wyniki. '
                           'Nie ma odpowiedzi ocenianych jako dobre lub złe.')
print(cases.iloc[0].text_standard); print(cases.iloc[0].explanation)

## 5 · Zamrożenie, kontrakt i aplikacja

In [ ]:
# Opcjonalnie (rozszerzenie, np. własny interfejs): zapis przetestowanych funkcji do model.py.
# Aplikacja kursu nie potrzebuje tego pliku: opisy, predykcje i wyjaśnienia są w cases.csv.
EKSPORT_MODELU = False
if EKSPORT_MODELU:
    FUNCS = [pct, describe, outcomes, explanation, to_long, framing_index]
    CONSTS = ['SAFE', 'RISKY', 'CODE', 'MODEL_VERSION']
    src = ['"""model.py – wygenerowane z notatnika ścieżki D; nie edytuj ręcznie."""', 'import pandas as pd', '']
    src += [f'{n} = {globals()[n]!r}' for n in CONSTS] + [''] + [inspect.getsource(f) for f in FUNCS]
    open('model.py', 'w', encoding='utf-8').write('\n'.join(src))
    sys.path.insert(0, WORKDIR)
    if 'model' in sys.modules: del sys.modules['model']
    import model
    check('model.py generuje ten sam opis', model.describe(120, .25, 'L') == describe(120, .25, 'L'))

## Zamrożenie przed sesją
Komórka zapisuje `cases.csv` z opisami opcji i predykcjami obu modeli oraz `session_config.json` ze skrótem SHA-256. Skrót wpisz do dziennika **przed sesją**. W trybie `wlasne` notatnik nie nadpisze zamrożonych przypadków: jeśli po zamrożeniu zmienisz dane albo kod, komórka zatrzyma się z błędem. 

In [ ]:
new_csv = cases.to_csv(index=False).encode()
new_hash = hashlib.sha256(new_csv).hexdigest()
if not PROFILE_OK:
    raise RuntimeError(f'Profil nie pozwala na personalizację ({PROFILE_NOTE}). Nie zamrażamy przypadków i nie prowadzimy sesji.')
if TRYB == 'wlasne' and os.path.exists('session_config.json'):
    frozen = json.load(open('session_config.json', encoding='utf-8'))
    if frozen['frozen_sha256'] != new_hash:
        raise RuntimeError('Przypadki są już zamrożone, a nowe obliczenia dają inny cases.csv. Nie nadpisuję. '
                           'Jeśli zmiana nastąpiła PRZED sesją, usuń świadomie session_config.json i opisz to w dzienniku; '
                           'po sesji wymaga to nowej wersji modelu i nowego zestawu.')
    print('Zamrożenie bez zmian:', new_hash)
else:
    open('cases.csv', 'wb').write(new_csv)
    json.dump(dict(contract='D_v1', participant_id=PARTICIPANT, session_id=SESSION_ID, frozen_sha256=new_hash,
                   frozen_at=now_iso(), model_version=MODEL_VERSION, seed=20261020, instructor_pin=INSTRUCTOR_PIN,
                   profile_ok=PROFILE_OK, profile_note=PROFILE_NOTE, **SESSION_EXTRA),
              open('session_config.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    print('Zamrożono cases.csv | SHA-256:', new_hash)
FROZEN_HASH = file_sha256('cases.csv')

In [ ]:
cfg = json.load(open('session_config.json', encoding='utf-8'))
_cases = pd.read_csv('cases.csv')
missing = [c for c in ['order', 'pair_id', 'case_id', 'origin_frame', 'endowment', 'success_p', 'safe_kept', 'text_standard', 'text_gain', 'text_loss', 'explanation'] if c not in _cases.columns]
check('cases.csv: wymagane kolumny', not missing or print('Brakuje:', missing))
check('session_config.json: kontrakt i zgodny skrót cases.csv', cfg['contract'] == 'D_v1' and cfg['frozen_sha256'] == file_sha256('cases.csv'))
check('profil.csv istnieje', os.path.exists('profil.csv'))
check('pytania o rozumienie zapisane w konfiguracji', len(cfg.get('comprehension', [])) >= 1)

## Aplikacja i sesja z uczestnikiem
Komórka z kodem aplikacji jest zwinięta. **Nie edytuj jej.** Zapisuje plik `app.py` w folderze projektu.

**Przebieg sesji:**
1. **Etap standardowy:** uczestnik widzi każdą z 12 wersji w jednym opisie i wybiera A albo B (`initial`).
2. **Etap zbalansowany:** dopiero po wszystkich wyborach wstępnych widzi dla tych samych wersji oba opisy obok siebie, wyjaśnienie równoważności i swój wcześniejszy wybór. Potwierdza go albo zmienia (`final`).
3. Na końcu odpowiada na obowiązkowe pytania o rozumienie i daje jedną ocenę transparentności całej serii (1–7).

Widok prowadzącego (panel boczny, kod `INSTRUCTOR_PIN`) pokazuje kontrakt, zamrożone przypadki i log. Nie otwieraj go przy uczestniku. Odświeżenie strony nie gubi danych: aplikacja wznawia sesję od pierwszego niezapisanego kroku.

In [ ]:
#@title Kod aplikacji – gotowy, nie edytuj { display-mode: "form" }
APP_SOURCE = r'''"""Sesja walidacyjna – ścieżka D (framing: etap standardowy, potem zbalansowany).

Gotowa aplikacja kursu: zespół jej NIE edytuje. Aplikacja niczego nie przewiduje i nie rekomenduje opcji.
Etap 1 zbiera wszystkie wybory wstępne (initial) przy jednym opisie. Dopiero potem etap 2 pokazuje oba
równoważne opisy z wyjaśnieniem i zbiera wybory końcowe (final). Na końcu: pytania o rozumienie
i jedna ocena transparentności całej serii.

Uruchomienie:  streamlit run app.py -- --data <folder z plikami projektu>
"""
import argparse
import hashlib
import json
import os
import sys
from datetime import datetime, timezone

import pandas as pd
import streamlit as st

APP_VERSION = 'D_app_v1'
CONTRACT = 'D_v1'
REQUIRED_CASES = ['order', 'pair_id', 'case_id', 'origin_frame', 'endowment', 'success_p', 'safe_kept',
                  'text_standard', 'text_gain', 'text_loss', 'explanation']
CODE = {'A': 'SAFE', 'B': 'RISKY'}
LABEL = {'SAFE': 'A', 'RISKY': 'B'}


def data_dir():
    p = argparse.ArgumentParser()
    p.add_argument('--data', default='.')
    return os.path.abspath(p.parse_known_args(sys.argv[1:])[0].data)


DATA = data_dir()
path = lambda name: os.path.join(DATA, name)
now_iso = lambda: datetime.now(timezone.utc).isoformat(timespec='seconds')
sha256 = lambda p: hashlib.sha256(open(p, 'rb').read()).hexdigest()


def load_project():
    errors = []
    for name in ['cases.csv', 'profil.csv', 'session_config.json']:
        if not os.path.exists(path(name)):
            errors.append(f'Brak pliku {name} w folderze {DATA}. Uruchom notatnik do sekcji „Zamrożenie”.')
    if errors:
        return {}, None, errors
    cfg = json.load(open(path('session_config.json'), encoding='utf-8'))
    if cfg.get('contract') != CONTRACT:
        errors.append(f'session_config.json: kontrakt {cfg.get("contract")} zamiast {CONTRACT}')
    if len(cfg.get('comprehension', [])) < 1:
        errors.append('session_config.json: brak pytań o rozumienie (są obowiązkowe).')
    cases = pd.read_csv(path('cases.csv'))
    missing = [c for c in REQUIRED_CASES if c not in cases.columns]
    if missing:
        errors.append(f'cases.csv: brak kolumn {", ".join(missing)}')
    if sha256(path('cases.csv')) != cfg.get('frozen_sha256'):
        errors.append('cases.csv różni się od wersji zamrożonej (SHA-256 nie zgadza się z session_config.json).')
    if not errors:
        cases = cases.sort_values('order').reset_index(drop=True)
    return cfg, cases, errors


def read_log():
    if os.path.exists(path('validation.csv')) and os.path.getsize(path('validation.csv')) > 0:
        return pd.read_csv(path('validation.csv'), dtype={'session_id': str, 'initial': object, 'final': object, 'final_at': object})
    return pd.DataFrame(columns=['session_id', 'case_id', 'initial', 'final'])


def write_log(log):
    tmp = path('validation.csv.tmp')
    log.to_csv(tmp, index=False)
    os.replace(tmp, path('validation.csv'))


st.set_page_config(page_title='Wybory punktowe', page_icon='⚖️', layout='wide')
cfg, cases, errors = load_project()
SESSION = str(cfg.get('session_id', 'S01'))
SESSION_JSON = path(f'session_{SESSION}.json')

with st.sidebar:
    st.caption(f'{APP_VERSION} · kontrakt {CONTRACT}')
    pin = st.text_input('Kod prowadzącego', type='password')
    instructor = bool(cfg) and pin != '' and pin == str(cfg.get('instructor_pin'))

if instructor:
    st.title('Widok prowadzącego')
    for e in errors:
        st.error(e)
    if not errors:
        st.success(f'Kontrakt spełniony · cases.csv zgodny z zamrożeniem · SHA-256 {cfg["frozen_sha256"][:16]}…')
        st.dataframe(cases[['order', 'case_id', 'origin_frame', 'endowment', 'success_p', 'safe_kept']], hide_index=True)
    log = read_log()
    st.subheader(f'Log sesji {SESSION}')
    st.dataframe(log, hide_index=True)
    if os.path.exists(path('validation.csv')):
        st.download_button('Pobierz validation.csv', open(path('validation.csv'), 'rb').read(), 'validation.csv', 'text/csv')
    if os.path.exists(SESSION_JSON):
        st.json(json.load(open(SESSION_JSON, encoding='utf-8')))
    st.stop()

st.title(cfg.get('title', 'Wybory punktowe'))
if errors:
    st.error('Aplikacja nie jest gotowa do sesji. Poproś prowadzącego o sprawdzenie (widok prowadzącego w panelu bocznym).')
    st.stop()
if os.path.exists(SESSION_JSON):
    st.success('Sesja zakończona. Dziękujemy za udział!')
    st.caption('Dla prowadzącego: próbną sesję można powtórzyć komórką „↺ Próbna sesja od początku” w notatniku.')
    st.stop()

log = read_log()
mine = log[log.session_id.astype(str) == SESSION]
have_initial = set(mine.case_id)
have_final = set(mine.loc[mine['final'].notna(), 'case_id']) if 'final' in mine else set()
todo_initial = cases[~cases.case_id.isin(have_initial)]
todo_final = cases[~cases.case_id.isin(have_final)]

if 'started' not in st.session_state:
    st.session_state.started = len(mine) > 0
if not st.session_state.started:
    st.write(cfg['intro'])
    if st.button('Rozpocznij', type='primary'):
        st.session_state.started = True
        st.rerun()
    st.stop()

if len(todo_initial):                                    # ---- etap 1: jeden opis
    row = todo_initial.iloc[0]
    k = len(have_initial) + 1
    st.progress((k - 1) / (2 * len(cases)), text=f'Etap 1 · sytuacja {k} z {len(cases)}')
    with st.container(border=True):
        st.markdown(row.text_standard.replace(' A: ', '\n\n**A:** ').replace(' B: ', '\n\n**B:** '))

    def choose_initial(opt):
        rec = dict(participant_id=cfg['participant_id'], session_id=SESSION, order=int(row.order), pair_id=row.pair_id,
                   case_id=row.case_id, origin_frame=row.origin_frame, endowment=row.endowment, success_p=row.success_p,
                   safe_kept=row.safe_kept, initial=CODE[opt], initial_at=now_iso(), final=None, final_at=None,
                   app_version=APP_VERSION)
        write_log(pd.concat([read_log(), pd.DataFrame([rec])], ignore_index=True))

    b1, b2 = st.columns(2)
    b1.button('Wybieram A', key=f'iA_{row.case_id}', on_click=choose_initial, args=('A',), use_container_width=True)
    b2.button('Wybieram B', key=f'iB_{row.case_id}', on_click=choose_initial, args=('B',), use_container_width=True)
    st.stop()

if len(todo_final):                                      # ---- etap 2: oba opisy
    row = todo_final.iloc[0]
    k = len(have_final) + 1
    st.progress((len(cases) + k - 1) / (2 * len(cases)), text=f'Etap 2 · sytuacja {k} z {len(cases)}')
    st.write('Ta sama sytuacja opisana na dwa równoważne sposoby. Możesz potwierdzić albo zmienić wcześniejszy wybór.')
    c1, c2 = st.columns(2)
    with c1:
        with st.container(border=True):
            st.caption('Opis przez zachowanie punktów')
            st.markdown(row.text_gain.replace(' A: ', '\n\n**A:** ').replace(' B: ', '\n\n**B:** '))
    with c2:
        with st.container(border=True):
            st.caption('Opis przez utratę punktów')
            st.markdown(row.text_loss.replace(' A: ', '\n\n**A:** ').replace(' B: ', '\n\n**B:** '))
    st.info(row.explanation)
    prev = mine.set_index('case_id').loc[row.case_id, 'initial']
    st.write(f'Twój wcześniejszy wybór: **{LABEL[prev]}**')

    def choose_final(opt):
        lg = read_log()
        m = (lg.session_id.astype(str) == SESSION) & (lg.case_id == row.case_id)
        lg.loc[m, 'final'] = CODE[opt]
        lg.loc[m, 'final_at'] = now_iso()
        write_log(lg)

    b1, b2 = st.columns(2)
    b1.button('Wybieram A', key=f'fA_{row.case_id}', on_click=choose_final, args=('A',), use_container_width=True)
    b2.button('Wybieram B', key=f'fB_{row.case_id}', on_click=choose_final, args=('B',), use_container_width=True)
    st.stop()

st.progress(1.0, text='Wszystkie wybory zapisane')                     # ---- rozumienie i ocena
st.header('Dwa krótkie pytania i ocena')
answers = {}
for q in cfg['comprehension']:
    row = cases[cases.pair_id == q['pair_id']].iloc[0]
    st.markdown(row.text_gain.replace(' A: ', '\n\n**A:** ').replace(' B: ', '\n\n**B:** '))
    answers[q['pair_id']] = st.number_input(q['question'], min_value=0, step=1, value=None, key=f'q_{q["pair_id"]}')
transparency = st.slider('Na ile przedstawienie obu opisów było dla Ciebie zrozumiałe? (1 = wcale, 7 = całkowicie)', 1, 7, 4)
ready = all(v is not None for v in answers.values())
if st.button('Zakończ sesję', type='primary', disabled=not ready):
    comp = [dict(pair_id=q['pair_id'], answer=int(answers[q['pair_id']]), correct=int(answers[q['pair_id']]) == int(q['correct']))
            for q in cfg['comprehension']]
    json.dump(dict(participant_id=cfg['participant_id'], session_id=SESSION, series_transparency=int(transparency),
                   comprehension=comp, finished_at=now_iso(), cases_sha256=cfg['frozen_sha256'], app_version=APP_VERSION),
              open(SESSION_JSON, 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
    st.rerun()
if not ready:
    st.caption('Odpowiedz na oba pytania, aby zakończyć.')
'''
open('app.py', 'w', encoding='utf-8').write(APP_SOURCE)
print('Zapisano app.py w', WORKDIR)

In [ ]:
#@title ▶ Uruchom aplikację w Colabie { display-mode: "form" }
URUCHOM = False  #@param {type:"boolean"}
import subprocess, socket, time
def _port_open(port):
    with socket.socket() as s:
        return s.connect_ex(('127.0.0.1', port)) == 0
if URUCHOM:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'streamlit'], check=True)
    if not _port_open(8501):
        subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', os.path.join(WORKDIR, 'app.py'),
                          '--server.port', '8501', '--server.headless', 'true',
                          '--server.enableCORS', 'false', '--server.enableXsrfProtection', 'false',
                          '--', '--data', WORKDIR],
                         stdout=open(os.path.join(WORKDIR, 'streamlit.log'), 'w'), stderr=subprocess.STDOUT)
        for _ in range(60):
            if _port_open(8501): break
            time.sleep(1)
    if IN_COLAB:
        from google.colab import output
        output.serve_kernel_port_as_window(8501, anchor_text='▶ Otwórz aplikację w nowej karcie')
    else:
        print('Aplikacja: http://localhost:8501')
else:
    print('Zaznacz URUCHOM i uruchom komórkę ponownie. Jeśli link nie działa, przeprowadź sesję lokalnie (komórka niżej).')

In [ ]:
#@title ↺ Próbna sesja od początku { display-mode: "form" }
PROBA_OD_POCZATKU = False      #@param {type:"boolean"}
TO_BYLA_TYLKO_PROBA = False    #@param {type:"boolean"}
import shutil
_sid = json.load(open('session_config.json', encoding='utf-8'))['session_id']
_files = [f for f in ['validation.csv', f'session_{_sid}.json'] if os.path.exists(f)]
if not PROBA_OD_POCZATKU:
    print('Po zakończonej sesji aplikacja pokazuje tylko podziękowanie. Aby powtórzyć próbną sesję, zaznacz PROBA_OD_POCZATKU i uruchom komórkę.')
elif not _files:
    print('Brak logu sesji: aplikacja zacznie od początku.')
elif TRYB == 'wlasne' and not TO_BYLA_TYLKO_PROBA:
    print("⚠ Tryb 'wlasne': log może zawierać prawdziwą sesję uczestnika. Jeśli to była tylko próba przed sesją, "
          'zaznacz też TO_BYLA_TYLKO_PROBA. Po sesji z uczestnikiem logu nie przenosimy.')
else:
    _dst = os.path.join('proby', datetime.now().strftime('%Y%m%d_%H%M%S'))
    os.makedirs(_dst, exist_ok=True)
    for f in _files:
        shutil.move(f, os.path.join(_dst, f))
    print('Przeniesiono', ', '.join(_files), 'do', os.path.join(WORKDIR, _dst), '(nic nie usunięto). Odśwież kartę aplikacji.')
    if TRYB == 'wlasne':
        print('Zapisz w dzienniku: data, kto prowadził próbę i że log próby przeniesiono do', _dst)

**Sesja lokalna (zalecana przy uczestniku).** Colab może się rozłączyć w trakcie sesji. Pewniej jest uruchomić aplikację na laptopie:
1. Uruchom komórkę poniżej i pobierz `pakiet_sesji_D.zip` (`app.py`, `profil.csv`, `cases.csv`, `session_config.json`).
2. Na laptopie: `pip install streamlit pandas`, potem w rozpakowanym folderze `streamlit run app.py -- --data .`
3. Po sesji skopiuj `validation.csv` i `session_S01.json` do folderu projektu na Dysku i uruchom analizę w trybie `wlasne`.

In [ ]:
import zipfile
with zipfile.ZipFile('pakiet_sesji_D.zip', 'w') as z:
    for f in ['app.py', 'profil.csv', 'cases.csv', 'session_config.json']:
        z.write(f)
print('Zapisano', os.path.join(WORKDIR, 'pakiet_sesji_D.zip'))
if IN_COLAB:
    from google.colab import files
    files.download('pakiet_sesji_D.zip')

## 6 · Wyniki walidacji (rozdz. 9)
- **Rozbieżność par przed:** pary z różnym `initial` w G i L, liczone tylko dla par kompletnych. To porównanie z profilowaniem.
- **Rozbieżność par po:** to samo dla `final`. To **kontrola**: obie wersje pokazują ten sam ekran.
- **Udział rewizji:** wersje ze zmianą `initial` → `final` podzielone przez wersje z obiema odpowiedziami, z kierunkiem zmian.
- **Rozumienie:** liczba poprawnych odpowiedzi. **Transparentność:** jedna ocena 1–7 dla całej serii.

In [ ]:
def validation_metrics(log, session):
    # TODO: rozbieżność par przed i po (licznik, mianownik), rewizje i ich kierunek, rozumienie, transparentność
    raise NotImplementedError

INITIAL_P01 = {'N01_G': 'SAFE', 'N01_L': 'RISKY', 'N02_G': 'SAFE', 'N02_L': 'SAFE', 'N03_G': 'RISKY', 'N03_L': 'SAFE',
               'N04_G': 'RISKY', 'N04_L': 'RISKY', 'N05_G': 'SAFE', 'N05_L': 'RISKY', 'N06_G': 'SAFE', 'N06_L': 'SAFE'}
FINAL_P01 = {**INITIAL_P01, 'N01_L': 'SAFE', 'N03_L': 'RISKY'}

def synthetic_session():
    log = cases[['order', 'pair_id', 'case_id', 'origin_frame', 'endowment', 'success_p', 'safe_kept']].copy()
    log.insert(0, 'participant_id', PARTICIPANT); log.insert(1, 'session_id', SESSION_ID)
    log['initial'] = log.case_id.map(INITIAL_P01); log['initial_at'] = now_iso()
    log['final'] = log.case_id.map(FINAL_P01); log['final_at'] = now_iso(); log['app_version'] = 'synthetic'
    comp = [dict(pair_id=q['pair_id'], answer=q['correct'], correct=True) for q in COMPREHENSION]
    return log, dict(series_transparency=6, comprehension=comp)

_log, _ses = synthetic_session()
_m = validation_metrics(_log, _ses)
check('dane P01: przed 3/6, po 1/6 (kontrola), rewizje 2/12 (N01_L, N03_L)', (_m['discord_before'], _m['discord_after'], _m['revisions'], _m['revised_cases'])
      == ((3, 6), (1, 6), (2, 12), ['N01_L', 'N03_L']))
_x = _log.copy(); _x.loc[_x.case_id == 'N04_L', 'final'] = None
check('przerwanie: mianowniki maleją, brak nie jest SAFE', validation_metrics(_x, _ses)['revisions'] == (2, 11))

In [ ]:
check('cases.csv nie zmienił się od zamrożenia', file_sha256('cases.csv') == FROZEN_HASH)
validation, session = None, None
if TRYB == 'przyklad':
    validation, session = synthetic_session()          # odpowiedzi syntetyczne w formacie logu aplikacji
elif os.path.exists('validation.csv'):
    validation = pd.read_csv('validation.csv', dtype={'session_id': str, 'case_id': str})
    validation = validation[validation.session_id == SESSION_ID]
    if os.path.exists(f'session_{SESSION_ID}.json'):
        session = json.load(open(f'session_{SESSION_ID}.json', encoding='utf-8'))
    _frozen = pd.read_csv('cases.csv', dtype={'case_id': str})
    check('log zawiera wyłącznie zamrożone przypadki', set(validation.case_id) <= set(_frozen.case_id.astype(str)))
    if 'pred_personal' in validation:
        _m = validation.merge(_frozen, on='case_id', suffixes=('', '_f'))
        check('predykcje w logu zgodne z zamrożonymi', ((_m.pred_personal == _m.pred_personal_f) & (_m.pred_baseline == _m.pred_baseline_f)).all())
    print(f'Wczytano {len(validation)} zapisów z sesji {SESSION_ID}.', '' if session else '⚠ Brak pliku session_*.json.')
else:
    print('Brak validation.csv: najpierw przeprowadź sesję.')

In [ ]:
if validation is not None:
    m = validation_metrics(validation, session)
    print(f"Przed: {m['discord_before'][0]}/{m['discord_before'][1]}  |  po (kontrola): {m['discord_after'][0]}/{m['discord_after'][1]}  |  "
          f"rewizje: {m['revisions'][0]}/{m['revisions'][1]} (A→B: {m['safe_to_risky']}, B→A: {m['risky_to_safe']})  |  "
          f"rozumienie {m['comprehension'][0]}/{m['comprehension'][1]}  |  transparentność {m['transparency']}/7")
    check_example('P01: rewizje 2/12, rozumienie 2/2, transparentność 6', (m['revisions'], m['comprehension'], m['transparency']) == ((2, 12), (2, 2), 6), m)
    print('Demonstracja przed i po w jednej osobie: etap, ponowny kontakt z problemem i pamięć współwystępują.')

## 7 · Pytania do raportu
1. Czy FI = 0,4 oznacza, że 40% decyzji było błędnych?
2. Dlaczego rozbieżność par po etapie zbalansowanym nie może być głównym wynikiem?
3. Czy rewizja N03_L z A (SAFE) na B (RISKY) jest porażką systemu?